# Testing a Grammar with FsLexYacc

This is the F# counterpart of `Python/ANTLR/PureParser.ipynb`. The original notebook generated Python and Java ANTLR parsers to recognize arithmetic expressions. Here the same grammar is generated directly for F#; the returned typed tree also replaces the external GUI test rig. Build with `python FSharp/tools/build.py` before starting.

In [ ]:
// Checks remain active in both F# Interactive and the Jupyter kernel.
let checkNotebook condition =
    if not condition then failwith "Notebook check failed"


In [ ]:
#r "Generated/bin/Debug/net9.0/FsLexYacc.Runtime.dll"
#r "Generated/bin/Debug/net9.0/GrammarExamples.dll"
#load "../Support/Visualization.fsx"
open GrammarExamples
open Visualization
open System.IO

## Grammar

The layers `expr`, `product`, and `factor` encode precedence without ambiguous productions. The start rule requires the end of input. A valid prefix does not count as a successful parse.

```text
%{
open GrammarExamples
%}
%token <int> NUMBER
%token PLUS MINUS TIMES DIVIDE LPAREN RPAREN EOF
%start main
%type <GrammarExamples.Expression> main
%%
main: expr EOF { $1 }
expr:
  | expr PLUS product { Binary ("+", $1, $3) }
  | expr MINUS product { Binary ("-", $1, $3) }
  | product { $1 }
product:
  | product TIMES factor { Binary ("*", $1, $3) }
  | product DIVIDE factor { Binary ("/", $1, $3) }
  | factor { $1 }
factor:
  | LPAREN expr RPAREN { $2 }
  | NUMBER { Number $1 }
```

## Scanner

FsLex constructs tokens for whitespace, natural numbers, parentheses and arithmetic operators. An illegal character raises a positioned lexical error. FsYacc receives the generated token type, not an untyped token dictionary.

```text
{
module GrammarExamples.ExpressionLexer
open FSharp.Text.Lexing
open GrammarExamples.ExpressionParser
}
let digit = ['0'-'9']
rule token = parse
| [' ' '\t' '\r']+ { token lexbuf }
| '\n' { lexbuf.EndPos <- lexbuf.EndPos.NextLine; token lexbuf }
| '0' | ['1'-'9'] digit* { NUMBER (int (LexBuffer<char>.LexemeString lexbuf)) }
| '+' { PLUS }
| '-' { MINUS }
| '*' { TIMES }
| '/' { DIVIDE }
| '(' { LPAREN }
| ')' { RPAREN }
| eof { EOF }
| _ { failwithf "Illegal character %A at line %d, column %d" (LexBuffer<char>.LexemeString lexbuf) lexbuf.StartPos.Line (lexbuf.StartPos.Column+1) }
```

In [ ]:
let parseString text =
    try Ok (Parse.expression text)
    with error -> Error error.Message

let valid = parseString "1 + 2 * 3 - 4"
checkNotebook (Result.isOk valid)
printfn "%A" valid

The second `*` in the next expression cannot begin a factor, and `<` is not a token in the scanner. Both examples return errors so the whole notebook can continue after demonstrating them.

In [ ]:
let syntaxError = parseString "1 + 2 * 3 ** 4"
let lexicalError = parseString "1 < 2"
checkNotebook (Result.isError syntaxError && Result.isError lexicalError)
printfn "%A\n%A" syntaxError lexicalError

## Inspect a tree

Semantic values live in the F# union `Expression`. No Java process or window is needed to inspect their structure.

In [ ]:
let rec toTree expression =
    match expression with
    | Number n -> Leaf (string n)
    | Binary (operator, left, right) -> Node (operator, [toTree left; toTree right])
let tree = Parse.expression "1+2*3-4"
checkNotebook (tree = Binary ("-", Binary ("+", Number 1, Binary ("*", Number 2, Number 3)), Number 4))
let dot = tree |> toTree |> treeToDot
printfn "%s" dot

In [ ]:
Microsoft.DotNet.Interactive.Kernel.display(Microsoft.DotNet.Interactive.Kernel.HTML(renderDot dot), "text/html") |> ignore

Generated files are build products. Editing the grammar and rerunning the build is sufficient; the notebook does not delete files from the working directory.